# Qwen2.5-Coder-7B QLoRA Fine-tuning (Unsloth)

Trains `Qwen/Qwen2.5-Coder-7B` with QLoRA using **Unsloth** instead of plain `transformers+peft+trl`, on the rejection-sampled dataset produced by `generate_rft_dataset.ipynb` (`rft_dataset.jsonl`, fields: `instruction`, `output`).

**Why Unsloth here:** the old `fine-tune-qwen.ipynb` (vanilla peft/trl, `batch_size=1 x grad_accum=16`) took >12h for just 300 steps on Kaggle's free GPU and never finished a session. Unsloth's fused Triton kernels + patched attention/LoRA typically give ~1.5-2x training speedup and ~30-50% lower VRAM on 7B QLoRA, on top of a dataset that is now a few hundred examples instead of the full generic Magicoder set — both changes target the same session-timeout problem from different angles.

See the chat response for the runtime estimate and pros/cons discussion — not duplicated here to avoid the estimate going stale in the notebook.

In [ ]:
# Pin versions deliberately: unsloth moves fast and unpinned installs on Kaggle
# frequently break on transformers/peft/trl version mismatches.
!pip install -q --upgrade pip
!pip install -q "unsloth==2024.11.9" "unsloth_zoo"
!pip install -q --no-deps "trl==0.12.0" "peft==0.13.2" "accelerate==1.1.1" "bitsandbytes==0.44.1"

## 1. Imports & Login

In [ ]:
import os
import json
import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

try:
    wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = wandb_api_key
    REPORT_TO = "wandb"
except Exception:
    REPORT_TO = "none"

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B"
HUB_MODEL_NAME = "hoangnh39/qwen2.5-coder-7b-mbpp-rft"
DATASET_PATH = "rft_dataset.jsonl"   # local file produced by generate_rft_dataset.ipynb,
                                       # or swap for load_dataset("hoangnh39/qwen25coder-mbpp-rft")
MAX_SEQ_LENGTH = 1024

## 2. Load Base Model + Attach LoRA (Unsloth)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto-detect bf16/fp16 support
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,                       # 0 is required for Unsloth's fused kernels
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth", # Unsloth's checkpointing variant, lower VRAM than vanilla
    random_state=3407,
)
print(model.print_trainable_parameters())

## 3. Load & Format Dataset (rejection-sampled, from `generate_rft_dataset.ipynb`)

In [ ]:
records = [json.loads(line) for line in open(DATASET_PATH, encoding="utf-8")]
print(f"Loaded {len(records)} training examples "
      f"({sum(r['canonical_fallback'] for r in records)} canonical fallbacks)")

def to_text(example):
    return {"text": f"{example['instruction']}{example['output']}{tokenizer.eos_token}"}

dataset = Dataset.from_list(records).map(to_text)
print(dataset[0]["text"][:500])

## 4. Train

In [ ]:
train_args = SFTConfig(
    output_dir="./checkpoints",
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    num_train_epochs=3,              # dataset is small (few hundred examples) -> multiple epochs needed
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,    # effective batch size 16
    optim="adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_grad_norm=0.3,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_steps=50,
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    report_to=REPORT_TO,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=train_args,
)

trainer_stats = trainer.train()
print(trainer_stats)

## 5. Save / Push Adapter

In [ ]:
model.push_to_hub(HUB_MODEL_NAME, private=True)
tokenizer.push_to_hub(HUB_MODEL_NAME, private=True)
print(f"Adapter pushed to https://huggingface.co/{HUB_MODEL_NAME}")

## 6. Smoke Test (sanity check, not a benchmark)

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = """You are an expert Python programmer. Write the implementation for the following problem.
Do not output anything else but the code block.

Problem:
def is_palindrome(s: str) -> bool:
    \"\"\" Check if a string is a palindrome. \"\"\"

Code:
"""
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=200, temperature=0.2, do_sample=True, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))